In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Multi-agent patterns in the Cymbal Beauty store agent

| | |
|-|-|
| Author(s) | [Matt Robinson](https://github.com/mr394729) |

> **This copy keeps the output of one complete run** (24 September 2026, in a test namespace), so you can read what each cell prints even if a cell fails for you. Your numbers and wording will differ where a model answers. To start clean, choose **Edit > Clear Outputs of All Cells** in JupyterLab, or run `jupyter nbconvert --clear-output --inplace <notebook>`.

## Overview

### Agent Development Kit (ADK)

[Agent Development Kit](https://google.github.io/adk-docs/) (ADK) is an open-source framework for building agents. An ADK agent is a model, a set of tools and instructions. Agents can also contain other agents, and ADK decides how control passes between them.

### Multi-agent patterns

When one agent has too many tools or too many jobs, you split it. ADK gives you a small number of ways to combine agents, and each one has a name, such as *agent as a tool*, *coordinator and dispatcher* or *parallel fan-out*. Naming the pattern makes an agent easier to review, evaluate and govern.

### The store agent

The store agent in this repository helps a store manager and their team run the day. A coordinator agent holds the conversation. It can consult three specialists (shelf availability, team coverage and loss prevention), hand the conversation to a coaching agent, run a fixed workflow that writes the opening plan, and pass task requests to an agent that asks for approval before it writes anything.

<img width="60%" src="../docs/diagrams/store-agent-architecture.png" alt="Architecture of the Cymbal Beauty store agent" />

### Objectives

In this tutorial, you will learn how the store agent is put together and which multi-agent pattern each part uses.

You will complete the following tasks:

- Build the agent tree from the repository's code and read how each sub-agent joins in
- Run the agent locally and see a specialist consulted as a tool
- Run the opening plan and see three readers run in parallel before one writer
- Build a two-step sequential pipeline of your own

### Costs

This tutorial uses billable components of Google Cloud:

- Gemini on Vertex AI
- BigQuery

Learn about [Vertex AI pricing](https://cloud.google.com/vertex-ai/pricing) and [BigQuery pricing](https://cloud.google.com/bigquery/pricing), and use the [Pricing Calculator](https://cloud.google.com/products/calculator/) to generate a cost estimate based on your projected usage.

## Get started

### Set Google Cloud project information

This notebook runs against the store data you loaded in the previous notebook, in your own namespace. Set your project ID and the namespace you chose during setup.

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [2]:
import os
import sys
from pathlib import Path

PROJECT_ID = "[your-project-id]"  # @param {type: "string"}
WORKSHOP_NAMESPACE = "[your-namespace]"  # @param {type: "string"}

if PROJECT_ID == "[your-project-id]":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "")
if WORKSHOP_NAMESPACE == "[your-namespace]":
    WORKSHOP_NAMESPACE = os.environ.get("WORKSHOP_NAMESPACE", "")
if not PROJECT_ID or not WORKSHOP_NAMESPACE:
    raise ValueError("Set PROJECT_ID and WORKSHOP_NAMESPACE above.")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["WORKSHOP_NAMESPACE"] = WORKSHOP_NAMESPACE
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["STORE_OPS_ENV"] = "dev"

# The agent's code lives one folder up from this notebook
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

### Import libraries

In [3]:
import logging
import warnings

# Keep the notebook output to the agent's own events: ADK marks experimental features with a
# UserWarning, and the Gen AI SDK logs a note whenever a response mixes text and tool calls.
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("google_genai").setLevel(logging.ERROR)

In [4]:
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.runners import InMemoryRunner
from google.genai import types

from agents.cymbal_store_ops.agent import create_app

## Look at the agent tree

`create_app()` is the same function the deployed agent uses. It builds the coordinator and all of its sub-agents. Building the tree does not call the model.

The deployed app also prints every event to its log; `log_events=False` keeps that out of the notebook.

In [5]:
app = create_app(log_events=False)
root_agent = app.root_agent

print(f"Coordinator: {root_agent.name}")
for sub_agent in root_agent.sub_agents:
    print(f"  {sub_agent.name:26} mode={sub_agent.mode}")

Coordinator: store_manager_agent
  inventory_excellence       mode=single_turn
  associate_orchestration    mode=single_turn
  loss_prevention            mode=single_turn
  associate_development      mode=chat
  store_tasks                mode=task


The `mode` setting decides how a sub-agent joins the conversation:

- `single_turn`: the coordinator calls the sub-agent like a tool. The sub-agent answers one question and hands back. This is the **agent as a tool** pattern.
- `chat`: the coordinator hands the whole conversation over. This is the **coordinator and dispatcher** pattern.
- `task`: the sub-agent works through a job and can stop to ask the user something, such as an approval. This is the **human in the loop** pattern.

A `single_turn` or `task` sub-agent appears in the coordinator's tool list, next to its ordinary Python tools. Here the store reads are Python functions that query BigQuery with your credentials; in the deployed agent the same tools, with the same names, are served by the store's MCP server (notebook 05):

In [6]:
[getattr(tool, "name", getattr(tool, "__name__", "")) for tool in root_agent.tools]

['identify_demo_user',
 'workshop_clock',
 'get_my_work',
 'complete_my_task',
 'report_my_task_blocker',
 'get_merchandising_work',
 'get_guest_product_options',
 'get_guest_feedback',
 'search_products',
 'get_product_details',
 'daily_briefing',
 'describe_store_data',
 'query_store_data',
 'deliver_store_report',
 'get_store_inventory_summary',
 'list_store_inventory',
 'get_product_stock',
 'get_bopis_demand',
 'get_replenishment_status',
 'get_inventory_context',
 'get_shrink_signals',
 'get_loss_controls',
 'get_task_history',
 'get_end_of_day_metrics',
 'create_end_of_day_dashboard',
 'policy_lookup',
 'inventory_excellence',
 'associate_orchestration',
 'loss_prevention',
 'store_tasks']

The three `single_turn` specialists and `store_tasks` are at the end of the list. `associate_development` is not there: the coordinator transfers the conversation to it instead of calling it.

Look at one specialist. It has its own instruction and its own tools, and the coordinator only sees its name and description:

In [7]:
inventory = next(a for a in root_agent.sub_agents if a.name == "inventory_excellence")

print(inventory.description)
print()
print([tool.__name__ for tool in inventory.tools])

Analyzes inventory decisions and dependencies across stock, reservations, pickup commitments, locations, inbound supply and existing work. Returns specialist findings for the requested scope.

['get_osa_exceptions', 'check_store_stock', 'find_nearby_stock', 'get_bopis_demand', 'get_replenishment_status', 'get_task_status', 'get_stock_location', 'get_store_inventory_summary', 'list_store_inventory', 'query_store_data', 'get_inventory_context', 'describe_store_data']


## Run the agent locally

An ADK `Runner` sends a message to the agent and returns a stream of events: model responses, tool calls, tool results and the final answer. `InMemoryRunner` keeps the session in memory, which is enough for a notebook.

The agent reads who is signed in from the session state. Start a session as Dana, the manager of store S-014:

In [8]:
runner = InMemoryRunner(app=app)

session = await runner.session_service.create_session(
    app_name=app.name,
    user_id="dana",
    state={
        "user:user_id": "U-M014",
        "user:store_id": "S-014",
        "user:role": "store_manager",
        "user:first_name": "Dana",
    },
)

Define a helper that sends one message and prints what happened: each tool call, each hand-over to another agent, and the final answer.

In [9]:
async def ask(question: str) -> None:
    """Send one message and print the tool calls, transfers and final answer."""
    message = types.Content(role="user", parts=[types.Part(text=question)])
    async for event in runner.run_async(
        user_id=session.user_id, session_id=session.id, new_message=message
    ):
        for call in event.get_function_calls():
            print(f"[{event.author}] calls {call.name}")
        if event.actions and event.actions.transfer_to_agent:
            print(f"[{event.author}] hands over to {event.actions.transfer_to_agent}")
        if event.is_final_response() and event.content and event.content.parts:
            text = "".join(part.text or "" for part in event.content.parts if not part.thought)
            if text:
                print(f"\n{text}")

### Agent as a tool

Ask a staffing question. The coordinator does not have the roster tools itself, so it consults the `associate_orchestration` specialist, gets one answer back and writes the reply. This turn takes 30 to 45 seconds:

In [10]:
await ask("How should we cover pickup orders around Priya's break this morning?")

[store_manager_agent] calls associate_orchestration


[associate_orchestration] calls get_shift_roster
[associate_orchestration] calls get_pickup_workload
[associate_orchestration] calls get_coverage_requirements


[associate_orchestration] calls get_traffic_and_backlog



### Current Workload & Break Schedule
* **Store Time:** 9:00 AM (America/Chicago).
* **Priya (A-1004):** Shift 9:00 AM – 1:00 PM; scheduled 15-minute break is **10:30 AM – 10:45 AM**.
* **Pickup Queue:** 9 orders (13 units). 
  * Earliest order deadline: **9:30 AM** (`BO-000651`).
  * Latest current promise: **11:00 AM** (`BO-000658` and `BO-000659`).
  * Uninterrupted workload estimate: **54 minutes** (at 6 minutes/order).

---

### Feasibility Around the Break
* **Before the Break (9:00 AM – 10:30 AM):** 
  * If Priya starts uninterrupted picking at 9:00 AM, the first order (`BO-000651`) is estimated ready around **9:06 AM** (well within its 9:30 AM promise), and all 9 orders are estimated to finish by **9:54 AM**.
  * The queue has **24 minutes of schedule slack** (latest uninterrupted start is 9:24 AM to finish by 10:18 AM).
  * Consequently, the entire current queue can be completed before Priya's 10:30 AM break under continuous picking assumptions.

* **During the Break (10:30 A


Priya's break is scheduled from 10:30 AM to 10:45 AM. The current pickup queue has 9 orders (13 units) with deadlines between 9:30 AM and 11:00 AM, totaling about 54 minutes of picking work. If Priya starts picking at 9:00 AM, she is on track to clear all 9 existing orders around 9:54 AM, well before her break.

To cover new incoming orders and guest pickups:
• Priya (A-1004) handles primary picking from 9:00 AM until 10:30 AM.
• Jordan (A-1000) steps in for pickup coverage from 10:30 AM to 10:45 AM during Priya's break.
• Priya resumes pickup duties when she returns at 10:45 AM.


The coordinator made one call, to `associate_orchestration`, and the specialist read the roster, the pickup workload, the coverage requirements and the traffic. The output shows two answers: first the specialist's findings, then the coordinator's shorter reply written from them.

In this run: Priya's break is 10:30 to 10:45 AM, the queue is 9 orders with 13 units and the first is promised for 9:30 AM, and Jordan covers pickups during the break. The wording and the specialist's tool choices differ from run to run; the numbers should match.

### Parallel fan-out inside a workflow

Now ask for the opening plan. The coordinator calls one tool, `daily_briefing`. Inside that tool is a fixed workflow: a `ParallelAgent` runs three readers (inventory, coverage and loss) at the same time, then one `LlmAgent` writes the plan from what they found.

The order never changes, so it is written in code rather than left to the model. That saves three model calls on every opening. The opening takes about 15 seconds.

In [11]:
await ask("Morning. Just opened up. What should I be on top of first?")

[store_manager_agent] calls daily_briefing



1. **Assign pickup queue fulfillment** — Store S-014 has 9 pending pickup orders totaling 13 units, with the earliest promise due at 9:30 AM for Lumière Hydra Cream and the full queue window ending at 1:00 PM. Priya and Jordan are unassigned candidates covering the window. Under the snapshot scenario starting at 9:00 AM, the estimated continuous queue finish is 9:54 AM; under the latest-uninterrupted-start scenario starting at 9:24 AM, estimated queue finish is 10:18 AM with zero late orders. Recommendation: assign Priya to begin picking at 9:00 AM, accounting for her scheduled break at 10:30 AM.

2. **Pick and replenish skincare cream** — Lumière Hydra Cream (P-0101) has an empty shelf on fixture SK-04 and 7 units in backroom bay B2, with 4 units reserved across 3 pickup orders and inbound stock delayed. Merchandising directive MD-014-01 is waiting on stock and due at 10:00 AM. Recommendation: pick 4 units from backroom bay B2 for pickup orders BO-000651, BO-000652, and BO-000653, th

Only one call is printed: the three readers and the writer run inside `daily_briefing`, so their events do not reach this helper. The plan has three items: the pickup queue (9 orders, 13 units, first due 9:30 AM), Lumière Hydra Cream (empty shelf, 7 units in the backroom, 4 of them reserved) and the locked-case latch for Noir Velvet Eau de Parfum. The wording differs from run to run.

Look at the workflow that ran behind that tool:

In [12]:
from agents.cymbal_store_ops.sub_agents.daily_briefing import make_daily_briefing

briefing = make_daily_briefing(model="gemini-3.8-flash")
for step in briefing.sub_agents:
    children = [child.name for child in getattr(step, "sub_agents", [])]
    print(f"{type(step).__name__:16} {step.name:18} {children or ''}")

ParallelAgent    signals            ['briefing_inventory', 'briefing_coverage', 'briefing_shrink']
LlmAgent         plan_writer        
BriefingFinalize briefing_finalize  


The third step, `briefing_finalize`, is plain code with no model call. It checks the writer's draft against a schema and adds the store and the store time.

### Try other patterns

Ask for coaching to see a `chat` hand-over, or ask for a task to see the approval step. The agent stops and waits for approval; this notebook does not approve anything.

Try these example phrases:

```
Build a short coaching conversation for Noor using recent picking activity
Create a backroom check for the Lumière Hydra Cream
```

In [13]:
await ask("Build a short coaching conversation for Noor using recent picking activity")

[store_manager_agent] calls transfer_to_agent
[store_manager_agent] hands over to associate_development


[associate_development] calls get_coaching_context
[associate_development] calls get_learning_options



Here is a structured coaching conversation plan for Noor (A-1007), grounded in yesterday's (October 2) picking activity:

1. **Open with observations and recognition:**
   * Review the October 2 shift: Noor completed 23 order-line picks, with 12 completed within the 8-minute target and an overall median pick duration of 11 minutes.
   * Note that delays were primarily tagged with stock-location searches (7 picks) and floor guest interruptions (4 picks), which can overlap during a shift.

2. **Explore workflow obstacles:**
   * Ask Noor how locating stock felt during the shift and whether discrepancy between floor fixtures and backroom bays contributed to search delays.
   * Discuss how guest inquiries on the floor impacted pacing and whether stepping away from active pick paths felt manageable.

3. **Targeted development support:**
   * Since Noor has completed BOPIS Fundamentals, recommend the 8-minute refresher activity **Find stock across selling and backstock locations (LEARN-LOC-

Here the coordinator called `transfer_to_agent`, and the coaching agent took over the conversation. It read Noor's coaching context and learning options and built the plan from her picking on 2 October. The wording differs from run to run.

## Build a sequential pipeline

The store agent does not use every pattern. Here is one it does not use, built from scratch in a few lines: a `SequentialAgent` runs its sub-agents in order, and each one passes its result to the next through the session state (`output_key`).

In [14]:
drafter = LlmAgent(
    name="drafter",
    model="gemini-3.8-flash",
    instruction="Write a two-sentence shift-start note for a beauty store team about restocking the skincare fixture.",
    output_key="draft",
)

editor = LlmAgent(
    name="editor",
    model="gemini-3.8-flash",
    instruction="Rewrite this note in plain, friendly language under 30 words:\n\n{draft}",
    output_key="final_note",
)

pipeline = SequentialAgent(name="note_pipeline", sub_agents=[drafter, editor])

pipeline_runner = InMemoryRunner(agent=pipeline, app_name="note_pipeline")
pipeline_session = await pipeline_runner.session_service.create_session(
    app_name="note_pipeline", user_id="dana"
)
async for event in pipeline_runner.run_async(
    user_id="dana",
    session_id=pipeline_session.id,
    new_message=types.Content(role="user", parts=[types.Part(text="Go")]),
):
    if event.is_final_response() and event.content:
        print(f"[{event.author}] {event.content.parts[0].text}\n")

[drafter] Good morning team, let’s make replenishing the skincare fixture our top priority this morning by pulling backstock to fill all empty slots. Please ensure every product is front-faced, shelves are wiped clean, and testers are fresh and ready for our guests.



[editor] Good morning! Let’s restock skincare first today: fill empty spots from the back, wipe shelves clean, face the products, and refresh the testers. Thanks so much, team!



The drafter wrote two sentences. The editor read them from the session state through `{draft}` and shortened them.

## Cleaning up

This notebook creates no cloud resources. The sessions lived in memory and end when you restart the kernel.

## What's next

- [Pattern pages](../docs/patterns/README.md): every pattern with its ADK documentation and code
- [ADK multi-agent systems](https://google.github.io/adk-docs/agents/multi-agents/)
- Next notebook: [Tools, reports and sessions in the store agent](03_tools_and_workflows.ipynb)